# Chapter 13 — Human-in-the-Loop and Escalation UX

*The handoff is its own design problem.*

## Objective

Run the same escalation three ways — `APPROVE` (the human overrides the gate), `DENY` (keep the failure), `DEFER` (end the loop in `escalated`). Inspect what the reviewer sees and what the audit records.

In [ ]:
from pydantic import BaseModel
from glassloop.core import BaseAgent, Finish, TaskSpec, ToolCall
from glassloop.governance import (
    GovernanceHarness, HumanDecision, HumanResponse, PlausibilityGate,
    PolicyEngine, ScriptedReviewer, SyntaxGate, pii_policy,
)
from glassloop.tools import GovernedToolExecutor, RiskLevel, Tool, ToolRegistry

## Scenario: a tool call that triggers PII escalation

In [ ]:
class EmailIn(BaseModel):
    to: str
    body: str
class EmailOut(BaseModel):
    success: bool

def send_email(to, body):
    return {'success': True}

tool = Tool(name='send_email', description='send email',
            input_schema=EmailIn, output_schema=EmailOut,
            risk=RiskLevel.HIGH, fn=send_email)

class OneShotAgent(BaseAgent):
    def propose_action(self, state):
        if state.step > 0:
            return Finish(output='done')
        return ToolCall(tool_name='send_email',
                        arguments={'to': 'x@x.com', 'body': 'contact a@b.com'})

def harness():
    r = ToolRegistry(); r.register(tool)
    engine = PolicyEngine([pii_policy])
    ex = GovernedToolExecutor(r, gates=[SyntaxGate(), engine.as_gate(), PlausibilityGate()])
    return GovernanceHarness(OneShotAgent(), ex)

## What the reviewer sees

An `EscalationRequest` is JSON-serializable. In a real system you would put it on a queue and a reviewer would pick it up.

In [ ]:
rev_approve = ScriptedReviewer(HumanResponse(decision=HumanDecision.APPROVE, note='confirmed safe'))
h = harness()
traj = h.run(TaskSpec(goal='test'), human_reviewer=rev_approve)

print('Reviewer received this request:\n')
print(rev_approve.requests[0].to_json(indent=2))

## APPROVE — the gate is overridden, the tool runs

The audit log still records that a gate escalated and that a human overrode it. Nothing is hidden.

In [ ]:
tool_step = next(r for r in traj.records if r.action.kind == 'tool_call')
print('success:', tool_step.observation['success'])
print('gates: ', [(g["gate"], g["decision"]) for g in tool_step.observation['gate_results']])

## DENY — keep the failure, the loop continues

In [ ]:
rev_deny = ScriptedReviewer(HumanResponse(decision=HumanDecision.DENY))
h = harness()
traj = h.run(TaskSpec(goal='test'), human_reviewer=rev_deny)
tool_step = next(r for r in traj.records if r.action.kind == 'tool_call')
print('success:', tool_step.observation['success'])
print('status:  ', traj.final_state.status)

## DEFER — end the loop in `escalated`, route to a human queue

In [ ]:
rev_defer = ScriptedReviewer(HumanResponse(decision=HumanDecision.DEFER, note='need legal'))
h = harness()
traj = h.run(TaskSpec(goal='test'), human_reviewer=rev_defer)
print('final status:', traj.final_state.status)
print('records:    ', [r.action.kind for r in traj.records])

## Anti-patterns flagged here

- Escalation that dumps a 50-page transcript on the human.
- Escalation without timeout.
- Resuming without re-validating state.

In [ ]:
# Self-check
assert len(rev_approve.requests) == 1
assert rev_approve.requests[0].reason
print('OK')